# Lab Kit L3 · Make it yours

**Video:** Lab Kit L3 on [Foundation Models for Robotics](https://www.youtube.com/@FoundationModelsForRobotics)
**You need:** Lab Kit L2. About 30 minutes.

So far the arm teleported: `set_q` placed it, nothing pushed back. Now we switch on physics. Motors take time to get somewhere, objects have mass, and the fingers can squeeze a sponge. Then you'll add your own object to the scene and record a demonstration, the raw material for every learning lesson later in the series.

In [ ]:
# Setup: works in Google Colab and locally (after `pip install -e .` in the repo root)
import os, sys
if "google.colab" in sys.modules:
    os.environ["MUJOCO_GL"] = "egl"
    if not os.path.exists("fmr-robot-learning-labs"):
        !git clone -q https://github.com/kandapagari/fmr-robot-learning-labs.git
    %cd fmr-robot-learning-labs
    !pip install -q -e .
import numpy as np, matplotlib.pyplot as plt, mediapy as media, mujoco
import fmr6


def show_video(frames, fps=30):
    """Play frames inline. Without ffmpeg (some local installs) show a strip of stills instead."""
    try:
        media.show_video(frames, fps=fps)
    except RuntimeError:
        media.show_images(frames[:: max(1, len(frames) // 5)], columns=5)

## 0 · Running on your own machine

```bash
git clone https://github.com/kandapagari/fmr-robot-learning-labs.git
cd fmr-robot-learning-labs
pip install -e ".[dev]"
python -m mujoco.viewer --mjcf fmr6/assets/fr3/scene_table.xml   # interactive 3D viewer
```

In the viewer, double-click a body to select it, then Ctrl + right-drag to push it. Open the *Control* panel on the right to move each motor with a slider. (On macOS use `mjpython -m mujoco.viewer …`.)

## 1 · Model and data

Under `fmr6.Robot` are two MuJoCo objects. `model` is everything that doesn't change: links, masses, joint limits, motors. `data` is everything that does: joint angles (`qpos`), velocities (`qvel`), motor commands (`ctrl`), time. `mujoco.mj_step(model, data)` advances the world by one timestep.

In [ ]:
robot = fmr6.Robot("table", arm="fr3")
m, d = robot.model, robot.data
print("timestep:", m.opt.timestep, "s →", round(1 / m.opt.timestep), "steps per simulated second")
print("qpos has", m.nq, "numbers: 7 arm joints, 2 fingers, then 7 each for the cup and sponge (position + quaternion)")
print("ctrl has", m.nu, "numbers: 7 joint targets + gripper (0 closed … 255 open)")

## 2 · Motors take time

Each FR3 joint has a position motor: set `d.ctrl[j]` to a target angle and the motor pushes toward it. Step the simulation and record how joint 4 follows a sudden 30° command.

In [ ]:
mujoco.mj_resetDataKeyframe(m, d, 0)            # back to Franka's home pose, fingers open
d.ctrl[:7] = robot.ready; d.ctrl[7] = 255
target = robot.ready[3] + np.radians(30)
t, q4 = [], []
steps = int(1.0 / m.opt.timestep)              # 1 simulated second
for i in range(steps):
    if d.time >= 0.1:
        d.ctrl[3] = target                     # command a jump at t = 0.1 s
    mujoco.mj_step(m, d)
    t.append(d.time); q4.append(d.qpos[robot.qadr[3]])
plt.figure(figsize=(7, 3))
plt.plot(t, np.degrees(q4), label="joint 4 (actual)")
plt.axhline(np.degrees(target), ls="--", c="k", lw=0.8, label="command")
plt.xlabel("time (s)"); plt.ylabel("degrees"); plt.legend(); plt.tight_layout(); plt.show()

The joint doesn't jump. It accelerates, overshoots slightly or not at all depending on the motor's stiffness and damping, and settles. A5 (control loops) is about exactly this curve.

## 3 · Pick up the sponge

A scripted pick: IK gives the joint targets, the motors move there over time, the gripper closes, the arm lifts. The sponge is 10 cm long and 6 cm wide, and the Franka Hand opens to 8 cm, so we grip it across its width with `grip_dir=(0, 1, 0)`.

In [ ]:
def move(q_target, gripper, seconds, frames=None, log=None):
    """Ramp the motor targets from where they are to q_target, stepping physics. gripper: 0 closed … 255 open."""
    start, n = d.ctrl[:7].copy(), int(seconds / m.opt.timestep)
    log_every, frame_every = round(0.02 / m.opt.timestep), round((1 / 30) / m.opt.timestep)
    for i in range(n):
        s = min(1.0, (i + 1) / (0.7 * n)); s = s * s * (3 - 2 * s)    # smooth start and stop
        d.ctrl[:7] = start + s * (q_target - start); d.ctrl[7] = gripper
        mujoco.mj_step(m, d)
        if log is not None and i % log_every == 0:                    # 50 Hz, like a real robot log
            log.append(np.concatenate([d.qpos[robot.qadr], [d.ctrl[7]], d.body("sponge").xpos]))
        if frames is not None and i % frame_every == 0:                # ~30 fps video
            frames.append(robot.render(480, 320, camera={"lookat": [0.66, -0.05, 0.42], "distance": 1.2, "azimuth": 150, "elevation": -28}))

mujoco.mj_resetDataKeyframe(m, d, 0); mujoco.mj_forward(m, d)  # reset, then recompute positions from qpos
sponge = d.body("sponge").xpos.copy()
above = robot.ik(sponge + [0, 0, 0.08], q0=robot.ready, grip_dir=(0, 1, 0))
grasp = robot.ik(sponge + [0, 0, 0.005], q0=robot.ready, grip_dir=(0, 1, 0))
mujoco.mj_resetDataKeyframe(m, d, 0); mujoco.mj_forward(m, d)  # ik() moved the arm geometrically; undo that

frames, log = [], []
move(above, 255, 1.5, frames, log)     # over the sponge, fingers open
move(grasp, 255, 1.0, frames, log)     # down
move(grasp, 0, 0.8, frames, log)       # squeeze
move(above, 0, 1.5, frames, log)       # lift
print(f"sponge height: {sponge[2]:.3f} m → {d.body('sponge').xpos[2]:.3f} m")
show_video(frames, fps=30)

## 4 · Add your own object

`mujoco.MjSpec` loads a scene as an editable description. Add a body, give it a free joint (so it can fall and slide) and a shape, compile, and hand the new model to `fmr6.Robot`.

In [ ]:
spec = mujoco.MjSpec.from_file(str(fmr6.model_path("table", arm="fr3")))
block = spec.worldbody.add_body(name="block", pos=[0.60, 0.15, 0.55])     # 15 cm above the table
block.add_freejoint()
block.add_geom(type=mujoco.mjtGeom.mjGEOM_BOX, size=[0.025, 0.025, 0.025], rgba=[0.2, 0.5, 0.9, 1], mass=0.1)
robot2 = fmr6.Robot(arm="fr3", model=spec.compile())
mujoco.mj_resetDataKeyframe(robot2.model, robot2.data, 0)
drop = []
for i in range(int(0.6 / robot2.model.opt.timestep)):                # 0.6 s: the block falls onto the table
    mujoco.mj_step(robot2.model, robot2.data)
    if i % 10 == 0:
        drop.append(robot2.render(400, 280, camera={"lookat": [0.66, -0.05, 0.42], "distance": 1.2, "azimuth": 150, "elevation": -28}))
print("block rests at z =", robot2.data.body("block").xpos[2].round(3), "m (table top is at 0.38 m)")
show_video(drop, fps=30)

## 5 · Save a demonstration

`log` from section 3 holds, 50 times a second, the joint angles, the gripper command and where the sponge was. That's a demonstration: what a learning robot is trained on. Save it; Season 2 trains a policy on files like this (in LeRobot's format, which A8 introduces).

In [ ]:
demo = np.array(log)
np.savez("demo_pick_sponge.npz", q=demo[:, :7], gripper=demo[:, 7], sponge=demo[:, 8:], hz=50)
print("saved", demo.shape[0], "steps =", demo.shape[0] / 50, "s of demonstration")
plt.figure(figsize=(7, 3))
plt.plot(np.arange(len(demo)) / 50, demo[:, 10], label="sponge height (m)")
plt.plot(np.arange(len(demo)) / 50, demo[:, 7] / 255 * 0.1 + 0.38, label="gripper (open ↑)")
plt.xlabel("time (s)"); plt.legend(); plt.tight_layout(); plt.show()

## 6 · Try it

1. Pick up the blue block from section 4 instead of the sponge. (Build the pick with `robot2`.) Which way should the fingers close for a cube?
2. Try gripping the sponge along its length (`grip_dir=(1, 0, 0)`). What happens, and why?
3. Make the pick twice as fast. At what speed does the sponge slip out?
4. Move the cup with physics: push it 10 cm across the table with the closed fingers.
5. Record five picks with the sponge in slightly different places (`d.qpos` of its free joint) and save each to its own file. You've just made a tiny dataset.

That's the kit. Every lesson's lab from here on uses these same tools.

In [ ]:
# your experiments here